# Usage

This notebook does simple analysis on ROOT file output from SuperCDMS SuperSim DMC simulations.

The particular simulation analyzed here is CUTE Tower 3 with Ba133 source, defined by `simulation_jobs/CUTE-T3_Ba133_12inch_DMC_tuned.mac`. And the main focus is on the Silicon detectors (detectors 2 and 5).

Main objectives:
- Estimate energy point of saturation in the Silicon detectors
- Create method for plotting energy sharing between channels

Requirements:
- Ensure you are using a SuperCDMS module Apptainer kernel to run this notebook
- *(Note: Only module release version V07-00 was tested during this notebook's development)*

## Imports

In [ ]:
# Loading ROOT files
import uproot

# DMC output access utilities
from dmc_utils import (
    list_detector_events,
    get_traces_for_events,
    flip_traces,
    normalize_traces
)

# Plotting
import matplotlib.pyplot as plt

## Load DMC data

In [ ]:
# Set path to DMC ROOT file
file_path = "/home/nevenac/projects/scdms-dmc/output/CUTE-T3_Ba133_12inch_DMC_1Mevents/combined.root"

In [ ]:
# Get and display detector-event index
index = list_detector_events(file_path, "G4SimDir/g4dmcTES", unique=True)

## Get TES traces for all events that occured in silicon detectors

In [ ]:
# Define silicon detector IDs
silicon_detectors = [2, 5]

# Load TES traces for events in silicon detectors
# Load traces for each silicon detector and store in a dictionary
traces_by_detector = {}
for det in silicon_detectors:
    if det in index:
        events = index[det]
        data = get_traces_for_events(file_path, events, det)
        # data is a dict: {101: {"Trace": [...], "ChanNum": [...], ...}, 102: {...}, ...}
        traces_by_detector[det] = data
    else:
        print(f"Detector {det} not found in detector-event index. Available detectors: {list(index.keys())}")

# Combine traces from all silicon detectors into a single list
# Flatten the dictionary values into a list of trace data dicts, this avoids overwriting keys (for any same event numbers across detectors), allowing us to keep all traces
all_events = [
    event_data
    for det_dict in traces_by_detector.values()
    for event_data in det_dict.values()
]

print(f"Total events loaded from silicon detectors: {len(all_events)}")

## Get total event energy proxies

In [ ]:
# Verify we have 12 traces per event
for evt in all_events:
    len_trace = len(evt["Trace"])
    if len_trace != 12:
        print(f"Warning: Number of Traces for event {evt["EventNum"]} in detector {evt["DetNum"]} = {len_trace}")
print("All events have 12 traces!")

In [ ]:
# Flip and normalize traces
for evt in all_events:
    traces = evt["Trace"]
    evt["Trace"] = normalize_traces(flip_traces(traces))

# For each event, integrate all 12 traces and sum them 
# This is a proxy for the total energy deposited in the detector for that event
for evt in all_events:
    evt["TraceIntegral"] = sum([sum(trace) for trace in evt["Trace"]])

# Plot histogram energy distribution

## Determine saturation point

In [6]:
# Method for determining if a trace is saturated

# Get percentage of bins that have full saturation (==1) for normalized traces
# Add up bins of all 12 channel traces to do this calculation

# Plot energy proxy versus fraction saturated

## Link energy proxy to actual Edep

In [ ]:
# Load g4dmcHits

# Filter trueEnergy or trueEdx by events that occurred in silicon detectors

# Plot energy proxy versus actual energy deposited (edep)

In [8]:
# Plot fraction saturated versus actual energy deposited (edep)